In [44]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from datetime import datetime
from datetime import timedelta
import pandas as pd
from shapely.geometry import Point, Polygon
from math import radians, sin, cos, sqrt, atan2
import numpy as np

In [45]:
stops_df = pd.read_csv("stops.txt")
trips_df = pd.read_csv("trips.txt")
stop_times_df = pd.read_csv("stop_times.txt")

In [46]:

'''
all columns from the GTFS dataframes

'trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence',
'pickup_type', 'drop_off_type', 'shape_dist_traveled', 'timepoint',
'route_id', 'service_id', 'trip_headsign', 'direction_id', 'block_id',
'shape_id', 'scheduled_trip_id', 'trip_short_name',
'wheelchair_accessible', 'bikes_allowed', 'at_street',
'corner_placement', 'heading', 'location_type', 'on_street',
'parent_station', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon',
'stop_name', 'stop_position', 'stop_timezone', 'stop_url',
'wheelchair_boarding', 'zone_id'

final result
- trip_headsign
- bus number
- bus stop order
- bus direction? some way to differentiate direction
- stop lat/lon
- wheelchair boarding
- bikes_allowed
- arrival and departure time
'''

# Merge relevant data
merged_df = pd.merge(stop_times_df, trips_df, on='trip_id')
merged_df = pd.merge(merged_df, stops_df, on='stop_id')

merged_df = merged_df[['trip_id','arrival_time', 'departure_time', 'stop_sequence','trip_headsign', 'direction_id','route_id',
       'wheelchair_accessible', 'bikes_allowed', 'stop_lat', 'stop_lon',
       'stop_name']]
merged_df = merged_df.sort_values(by=['arrival_time', 'departure_time'])

all_unique_stops = merged_df.drop_duplicates(subset=[
    'stop_sequence',
    'trip_headsign',
    'direction_id',
    'route_id',
    'wheelchair_accessible',
    'bikes_allowed',
    'stop_lat',
    'stop_lon',
    'stop_name'
])

# Extract unique bus stops with their coordinates and bus numbers
# unique_stops_routes = merged_df[['route_id', 'stop_name', 'stop_lat', 'stop_lon']].drop_duplicates()
# unique_stops = unique_stops_routes[[ 'stop_name', 'stop_lat', 'stop_lon']].drop_duplicates()
all_unique_stops['is_express'] = (all_unique_stops['route_id'] >= 800) & (all_unique_stops['route_id'] < 900)
all_unique_stops.head()

C:\Users\Martin\AppData\Local\Temp\ipykernel_29644\1204175623.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_unique_stops['is_express'] = (all_unique_stops['route_id'] >= 800) & (all_unique_stops['route_id'] < 900)


,trip_id,arrival_time,departure_time,stop_sequence,trip_headsign,direction_id,route_id,wheelchair_accessible,bikes_allowed,stop_lat,stop_lon,stop_name,is_express
144455,2961157_3041,03:43:00,03:43:00,1,20 ABIA Airport SB,1,20,1,1,30.313865,-97.664697,7000 Manor/Susquehanna,False
144456,2961157_3041,03:44:13,03:44:13,2,20 ABIA Airport SB,1,20,1,1,30.310003,-97.667482,6650 Manor/Loyola,False
144457,2961157_3041,03:45:12,03:45:12,3,20 ABIA Airport SB,1,20,1,1,30.308500,-97.671213,University Hills SB (Manor/Northeast),False
144458,2961157_3041,03:46:50,03:46:50,4,20 ABIA Airport SB,1,20,1,1,30.307941,-97.678194,6102 Manor/Wheless,False
144459,2961157_3041,03:47:27,03:47:27,5,20 ABIA Airport SB,1,20,1,1,30.306590,-97.680101,5900 Manor/Sweeney,False


In [47]:
len(all_unique_stops['route_id'].unique())

71

In [48]:
# Step 1: Get the max stop_sequence per bus and bus direction
trip_dir_max_seq =  (
    all_unique_stops
    .groupby(['route_id', 'direction_id', 'trip_id'])['stop_sequence']
    .max()
    .reset_index()
    .rename(columns={'stop_sequence': 'max_stop'})
)
len(trip_dir_max_seq['route_id'].unique())


71

In [49]:
# Step 3: Merge to get trip + route + max stop_sequence
trip_route_seq = pd.merge(trip_dir_max_seq, all_unique_stops, on='trip_id')
# Step 2: Drop the unwanted columns and rename
trip_route_seq = trip_route_seq.drop(columns=['direction_id_y', 'route_id_y']).rename(columns={'direction_id_x': 'direction_id', 'route_id_x': 'route_id'})  # Rename "_x" to original

In [50]:
len(trip_route_seq['route_id'].unique())


71

In [51]:
# Step 4: Find the trip with the highest stop_sequence per route
longest_trips = (
    trip_route_seq
    .sort_values('max_stop', ascending=False)  # Sort to prioritize trips with highest max_stop
    .drop_duplicates(['route_id', 'trip_headsign', 'direction_id'], keep='first')  # Keep first occurrence (longest)
    ['trip_id']  # Optional: Keep only needed columns
)

longest_trips

553      2962625_4813
382      2962570_5064
25       2959568_0299
1618     2962803_5379
183      2959677_0191
            ...      
2146     2960401_1024
4571    2967904_13211
4582    2967913_13202
2144     2960389_1012
4877    2976018_18634
Name: trip_id, Length: 157, dtype: object

In [52]:
all_unique_stops['trip_id']

144455     2961157_3041
144456     2961157_3041
144457     2961157_3041
144458     2961157_3041
144459     2961157_3041
              ...      
537176    2967774_13068
537177    2967774_13068
536957    2967774_13062
537178    2967774_13068
536958    2967774_13062
Name: trip_id, Length: 5288, dtype: object

In [53]:
# Step 5: Filter the original DataFrame to keep only these trips
all_unique_stops = all_unique_stops[all_unique_stops['trip_id'].isin(longest_trips)]

In [54]:
len(all_unique_stops['route_id'].unique())

71

In [55]:
all_unique_stops.to_csv("all_unique_stops.csv")

In [57]:
trips_df['route_id'].unique()

array([  1,  10, 103, 105, 111, 135, 142, 152, 171,  18,   2,  20, 201,
       214, 217, 228, 233, 237, 243, 271,   3,  30, 300, 310, 311, 315,
       318, 322, 323, 324, 325, 333, 335, 337, 339, 345, 350, 383, 392,
         4, 465, 466, 481, 483, 484, 485, 486, 490, 491, 492, 493,   5,
        50, 550, 640, 642, 656, 661, 663, 670, 672,   7, 800, 801, 803,
       837, 935, 980, 982, 985, 990])